# 🎨 Multimodal GenAI — Hands-On Lab## Vision, image generation, audio, video, multimodal embeddings, and multimodal RAG.**Setup:** `pip install openai numpy pillow`**Cost:** ~$0.50 to run everything**Programs:** Vision analysis, image generation (DALL-E 3), speech-to-text (Whisper), text-to-speech, multimodal embeddings (CLIP concept), multimodal RAG (tables + images as text).

In [ ]:
# SETUP
# pip install openai numpy
from openai import OpenAI
import json, time, base64, numpy as np

client = OpenAI()

def ask(prompt, model='gpt-4o-mini', temperature=0, system=None):
    messages = []
    if system: messages.append({'role':'system','content':system})
    messages.append({'role':'user','content':prompt})
    r = client.chat.completions.create(model=model, messages=messages, temperature=temperature)
    return r.choices[0].message.content.strip(), r.usage.total_tokens

def embed(texts):
    if isinstance(texts, str): texts = [texts]
    r = client.embeddings.create(model='text-embedding-3-small', input=texts)
    return [np.array(d.embedding) for d in r.data]

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

answer, _ = ask('Say Multimodal ready in 2 words.')
print(f'Setup complete! {answer}')

---# 1️⃣ Vision-Language Models (LLMs That Can See)**Analogy:** A brilliant assistant who was blind their entire life — can discuss anything in text but show them a photo and they say 'I cannot see.' Vision models gave this assistant perfect vision overnight.**What we build:** Image analysis via URL, structured data extraction from images, and image comparison.

In [ ]:
# VISION: LLMs that can SEE images
# GPT-4o can analyze photos, charts, screenshots, documents

# ---------- 1. Describe an image from URL ----------
print('VISION — IMAGE ANALYSIS')
print('=' * 60)

image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'

print('\n  Analyzing image from URL...')
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role':'user','content':[
        {'type':'text','text':'Describe this image in 2 sentences. What do you see?'},
        {'type':'image_url','image_url':{'url': image_url, 'detail':'low'}}
    ]}],
)
print(f'  Description: {response.choices[0].message.content}')
print(f'  Tokens used: {response.usage.total_tokens}')
print(f'  (detail:low = 85 tokens. detail:high = 1000+ tokens)')

# ---------- 2. Structured extraction from image ----------
print('\n\n  STRUCTURED EXTRACTION FROM IMAGE:')
print('  ' + '-' * 45)

response2 = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role':'user','content':[
        {'type':'text','text':'Extract as JSON: {subject, colors (list), format}. JSON only.'},
        {'type':'image_url','image_url':{'url': image_url}}
    ]}],
)
print(f'  Extracted: {response2.choices[0].message.content[:120]}')

# ---------- 3. Token cost comparison ----------
print('\n\n  IMAGE TOKEN COSTS:')
print('  ' + '-' * 45)
print('  detail:low  = 85 tokens   (~$0.00001) — use for simple images')
print('  detail:high = 1000+ tokens (~$0.0002) — use for complex charts')
print('  At 10K images/day:')
print('    Low detail:  $0.10/day')
print('    High detail: $2.00/day')
print('\n  Always use low detail unless you need to read small text or charts.')

---# 2️⃣ Image Generation (DALL-E 3)**Analogy:** Describing a painting to an artist: 'a cozy cabin at sunset with snow on the roof.' They paint exactly that. DALL-E 3 is that artist — text description in, image out.**What we build:** Generate images via API, compare quality settings, and show cost per image.

In [ ]:
# IMAGE GENERATION: Text description → image
# DALL-E 3 via API. No GPU needed.

print('IMAGE GENERATION — DALL-E 3')
print('=' * 60)

# Generate an image
print('\n  Generating image...')
start = time.time()

response = client.images.generate(
    model='dall-e-3',
    prompt='A cozy mountain cabin at sunset, warm light from windows, snow on the roof, cinematic photography style',
    size='1024x1024',
    quality='standard',  # 'standard' or 'hd'
    n=1,
)

elapsed = time.time() - start
image_url = response.data[0].url
revised_prompt = response.data[0].revised_prompt

print(f'  Generated in {elapsed:.1f}s')
print(f'  URL: {image_url[:80]}...')
print(f'  Revised prompt: {revised_prompt[:80]}...')
print(f'  (DALL-E 3 rewrites your prompt for better results)')

print('\n\n  COST COMPARISON:')
print('  ' + '-' * 45)
costs = [
    ('DALL-E 3 standard 1024x1024', '$0.040'),
    ('DALL-E 3 HD 1024x1024', '$0.080'),
    ('DALL-E 3 standard 1024x1792', '$0.080'),
    ('DALL-E 3 HD 1024x1792', '$0.120'),
    ('Stable Diffusion (your GPU)', '~$0.001'),
]
for name, cost in costs:
    print(f'    {name:35s} {cost}')

print('\n  DALL-E 3: zero setup, pay per image. Best prompt understanding.')
print('  Stable Diffusion: need GPU but 40x cheaper per image.')
print('  Crossover: ~1000 images/day self-hosted becomes cheaper.')

---# 3️⃣ Audio & Speech (Whisper + TTS)**Analogy:** Whisper gives the AI ears (speech → text). TTS gives it a voice (text → speech). Together: a phone agent that hears, thinks, and speaks.**What we build:** Text-to-speech demo + Whisper transcription config + voice agent pipeline design.

In [ ]:
# AUDIO: Text-to-Speech (TTS) + Speech-to-Text (Whisper)

print('AUDIO & SPEECH')
print('=' * 60)

# ---------- TEXT-TO-SPEECH ----------
print('\n  TEXT-TO-SPEECH (OpenAI TTS):')
print('  ' + '-' * 45)

tts_response = client.audio.speech.create(
    model='tts-1',        # or 'tts-1-hd' for higher quality
    voice='nova',          # alloy, echo, fable, onyx, nova, shimmer
    input='Hello! I am an AI voice agent. How can I help you today?',
)

# Save to file
with open('speech_output.mp3', 'wb') as f:
    for chunk in tts_response.iter_bytes():
        f.write(chunk)

import os
size = os.path.getsize('speech_output.mp3')
print(f'  Generated speech: speech_output.mp3 ({size:,} bytes)')
print(f'  Voice: nova (6 voices: alloy, echo, fable, onyx, nova, shimmer)')
print(f'  Cost: $15 per 1M characters')

# ---------- SPEECH-TO-TEXT (Whisper) ----------
print('\n\n  SPEECH-TO-TEXT (Whisper):')
print('  ' + '-' * 45)
print('  # To transcribe an audio file:')
print('  # audio_file = open("meeting.mp3", "rb")')
print('  # transcript = client.audio.transcriptions.create(')
print('  #     model="whisper-1", file=audio_file')
print('  # )')
print('  # print(transcript.text)')
print('  #')
print('  # Cost: $0.006 per minute (1 hour = $0.36)')
print('  # Supports: 100+ languages, accents, background noise')

# ---------- VOICE AGENT PIPELINE ----------
print('\n\n  VOICE AGENT PIPELINE:')
print('  ' + '-' * 45)
print('  User speaks')
print('    → Whisper (speech→text)     ~500ms')
print('    → LLM (thinks)              ~200ms (Groq) to ~2s (GPT-4)')
print('    → TTS (text→speech)          ~300ms')
print('  User hears response')
print()
print('  Total: 1-3 seconds (naive) | <1s (optimized with streaming)')
print('  Key: stream everything. Start TTS before LLM finishes.')
print()
print('  Frameworks: OpenAI Realtime API, LiveKit, Pipecat')

---# 4️⃣ Video Generation (Honest Assessment)**Analogy:** Self-driving cars in 2015 — impressive demos, real promise, but not ready for production in most scenarios. Video gen is there NOW. Know what it can and cannot do.**What we cover:** Current models, practical reality, and what works TODAY.

In [ ]:
# VIDEO GENERATION: Current landscape (honest assessment)

print('VIDEO GENERATION — STATE OF THE ART')
print('=' * 60)

models = [
    ('Sora (OpenAI)',       '60s', 'Limited',  'Most hyped. Not widely available.'),
    ('Runway Gen-3',        '10s', 'API+Web',  'Most production-ready today.'),
    ('Kling (Kuaishou)',    '10s', 'API',      'Competitive quality. Growing fast.'),
    ('Stable Video Diff',   '4s',  'Local',    'Open-source. Image-to-video.'),
]

print(f'\n  {"Model":20s} {"Max":5s} {"Access":10s} {"Notes"}')
print('  ' + '-' * 65)
for name, duration, access, notes in models:
    print(f'  {name:20s} {duration:5s} {access:10s} {notes}')

print('\n\n  REALITY CHECK:')
print('  ' + '-' * 45)
checks = [
    ('Generation speed', 'Minutes to hours per video', '🟡'),
    ('Quality consistency', 'Physics violations, morphing artifacts', '🔴'),
    ('Cost', 'Very GPU-intensive', '🔴'),
    ('Duration', '5-60 seconds typically', '🟡'),
    ('Production ready?', 'For short creative clips: yes', '🟢'),
    ('Long-form content?', 'Not yet reliable', '🔴'),
]
for what, status, emoji in checks:
    print(f'  {emoji} {what:25s} {status}')

print('\n\n  WHAT WORKS TODAY:')
print('  ✅ Short social media clips')
print('  ✅ Product animations')
print('  ✅ Concept visualization / mood boards')
print('  ✅ B-roll footage')
print()
print('  WHAT DOES NOT WORK YET:')
print('  ❌ Long-form polished content')
print('  ❌ Real-time generation')
print('  ❌ Precise control over output')
print()
print('  Timeline: 2-3 years from mainstream production readiness.')
print('  Track this space monthly — it evolves fast.')

---# 5️⃣ Multimodal Embeddings (Same Space for Text + Images)**Analogy:** A universal translator converting BOTH English and Japanese into the same secret code. If someone writes 'dog' in English and another draws a dog, they get similar code numbers. This lets you search images with text.**What we build:** Demonstrate the concept by embedding text descriptions of images and showing cross-modal similarity.

In [ ]:
# MULTIMODAL EMBEDDINGS: Text and images in the same number space
# CLIP puts text and images in the same vector space
# Here we demonstrate the concept using text embeddings

print('MULTIMODAL EMBEDDINGS (CLIP concept)')
print('=' * 60)

# In production: use CLIP to embed actual images
# Here: we embed text DESCRIPTIONS to show the concept

# Imagine these are image embeddings (in production, use CLIP on actual images)
image_descriptions = [
    'a golden retriever playing on a sunny beach',
    'a red Ferrari sports car on a racing track',
    'a plate of sushi with chopsticks on a wooden table',
    'a snowy mountain landscape at sunset',
    'a tabby cat sleeping on a cozy couch',
]

# These are text search queries
search_queries = [
    'cute dog playing outside',
    'fast luxury automobile',
    'Japanese food',
    'winter mountain scenery',
    'sleeping pet',
]

# Embed everything
img_embs = embed(image_descriptions)
query_embs = embed(search_queries)

print('\n  Cross-modal search: text query → find matching image\n')

for i, query in enumerate(search_queries):
    # Find best matching 'image'
    scores = [(cosine(query_embs[i], img_emb), desc) for img_emb, desc in zip(img_embs, image_descriptions)]
    scores.sort(key=lambda x: x[0], reverse=True)
    best_score, best_match = scores[0]
    
    print(f'  Query: "{query}"')
    print(f'  Best match [{best_score:.3f}]: "{best_match}"')
    print()

print('  HOW CLIP WORKS:')
print('  1. Image encoder: photo of dog → [0.82, -0.31, 0.55, ...]')
print('  2. Text encoder: "a photo of a dog" → [0.80, -0.33, 0.54, ...]')
print('  3. Similar meaning = similar numbers (even across modalities!)')
print()
print('  USE CASES: visual search, zero-shot image classification,')
print('  content moderation, e-commerce product matching.')
print()
print('  Models: CLIP (OpenAI), SigLIP (Google), Jina-CLIP (production)')
print('  pip install open-clip-torch')

---# 6️⃣ Multimodal RAG (Tables + Images + Audio → Searchable)**Analogy:** A regular library only catalogs books (text). A multimedia library also catalogs paintings (images), recordings (audio), and data tables. You want the universal one.**What we build:** Convert tables to markdown + describe images as text + transcribe audio. All become searchable in one vector store.

In [ ]:
# MULTIMODAL RAG: Everything becomes searchable text
# Strategy: convert ALL content types to text, embed together

print('MULTIMODAL RAG')
print('=' * 60)

# ---------- 1. TABLE → MARKDOWN ----------
print('\n  1. TABLE → MARKDOWN')
print('  ' + '-' * 45)

table = [
    ['Region', 'Q3 Revenue', 'Growth'],
    ['North America', '$45M', '+12%'],
    ['Europe', '$32M', '+8%'],
    ['Asia Pacific', '$28M', '+22%'],
    ['Latin America', '$12M', '+35%'],
]

md_table = '| ' + ' | '.join(table[0]) + ' |\n'
md_table += '|' + '|'.join(['---'] * len(table[0])) + '|\n'
for row in table[1:]:
    md_table += '| ' + ' | '.join(row) + ' |\n'

print(f'  {md_table}')
table_chunk = f'Q3 Regional Revenue:\n{md_table}'

# ---------- 2. IMAGE → TEXT DESCRIPTION ----------
print('  2. IMAGE → TEXT DESCRIPTION')
print('  ' + '-' * 45)

# In production: send actual image to GPT-4o
# response = client.chat.completions.create(
#     model='gpt-4o',
#     messages=[{'role':'user','content':[
#         {'type':'text','text':'Describe this chart for a search system.'},
#         {'type':'image_url','image_url':{'url':'data:image/png;base64,...'}}
#     ]}]
# )

image_desc = (
    'Bar chart showing Q3 2024 regional revenue. '
    'North America leads at $45M (+12%), followed by Europe $32M (+8%), '
    'Asia Pacific $28M (+22%), Latin America $12M (+35%). '
    'Asia Pacific shows fastest growth rate.'
)
print(f'  GPT-4o description: {image_desc[:60]}...')
print(f'  Cost: ~$0.01 per image')

# ---------- 3. AUDIO → TEXT ----------
print('\n  3. AUDIO → TEXT (Whisper)')
print('  ' + '-' * 45)
audio_transcript = 'CEO stated Q3 exceeded expectations with 22% growth in Asia Pacific driven by the new Enterprise product launch.'
print(f'  Whisper transcript: {audio_transcript[:60]}...')
print(f'  Cost: $0.006/minute')

# ---------- 4. UNIFIED SEARCH ----------
print('\n\n  4. UNIFIED SEARCH (all content types together)')
print('  ' + '-' * 45)

all_chunks = [
    ('TEXT', 'Standard refund policy is 30 days. Electronics: 15 days.'),
    ('TABLE', table_chunk),
    ('CHART', f'Chart: {image_desc}'),
    ('AUDIO', f'Earnings call: {audio_transcript}'),
    ('TEXT', 'Pro plan costs $49/month. Enterprise $199/month.'),
]

chunk_texts = [text for _, text in all_chunks]
chunk_types = [ctype for ctype, _ in all_chunks]
chunk_embs = embed(chunk_texts)

queries = [
    'Which region grew fastest in Q3?',
    'What did the CEO say about Asia?',
    'What is the refund policy?',
]

for query in queries:
    q_emb = embed(query)[0]
    scored = [(cosine(q_emb, ce), ct, txt[:50]) for ce, ct, txt in zip(chunk_embs, chunk_types, chunk_texts)]
    scored.sort(key=lambda x: x[0], reverse=True)
    
    print(f'\n  Query: "{query}"')
    for score, ctype, preview in scored[:2]:
        emoji = {'TEXT':'📝','TABLE':'📊','CHART':'📈','AUDIO':'🎙️'}[ctype]
        print(f'    [{score:.3f}] {emoji} {ctype:5s}: {preview}...')

print('\n\n  The query found content across ALL types — text, table, chart, audio.')
print('  Strategy: convert everything to text. 90% of the value. Zero infra changes.')
print('  For maximum accuracy: also send original images to GPT-4o at query time.')

---# 🏁 Summary — Multimodal Decision Guide| Modality | Tool | Cost | Use Case ||----------|------|------|----------|| **See images** | GPT-4o Vision | $0.00001-0.0002/img | OCR, charts, screenshots || **Generate images** | DALL-E 3 | $0.04-0.12/img | Marketing, product viz || **Hear audio** | Whisper | $0.006/min | Transcription, voice input || **Speak** | OpenAI TTS | $15/1M chars | Voice agents, narration || **Generate video** | Runway Gen-3 | $0.05-0.50/sec | Short creative clips || **Cross-modal search** | CLIP/SigLIP | Free (local) | Visual search, classification || **Multimodal RAG** | Convert-to-text | ~$0.01/item | Search across all content types |**Start simple:** Convert everything to text. Embed as text. Search as text. 90% of the value with zero infrastructure changes.